# End-to-End Brushstroke Benchmark

This notebook runs the full pipeline explicitly in cells:

1. Generate stylized images (90 runs)
2. Save per-run training loss curves
3. Compute deception score **after generation** from saved final images
4. Build aggregate tables and benchmark plots

Target setup:
- 10 content images
- 3 style images
- brushstrokes: 200, 1200, 2500
- fixed 100 optimization steps
- final output image per run (`brushstroke_result.png`)
- per-run training curves (`training_loss_curves.png`)

Before running, make sure deception assets are downloaded:

```bash
python3 download_sanakoyeu.py
```

Expected total runs: **90**.

In [1]:
import csv
import os
import time
from pathlib import Path

# Avoid slow/stuck font-cache writes to an unwritable home directory.
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.optim as optim
from IPython.display import display
from torchvision.utils import save_image

from deception_score import (
    compute_deception_rate,
    get_artist_labels,
    get_deception_paths,
    load_deception_model,
)
from losses import StyleTransferLosses, total_variation_loss, curvature_loss
from renderer import BrushStrokeRenderer
from utils import image_loader, pick_device

ROOT = Path.cwd()
CONTENT_DIR = ROOT / "images" / "content"
STYLE_MANIFEST = ROOT / "images" / "style_manifest.csv"

OUT_DIR = ROOT / "results" / "experiments_90_fixed100_notebook"
VGG_WEIGHTS = ROOT / "vgg_weights" / "vgg19_weights_normalized.h5"

IMG_SIZE = 512
NUM_STROKES_LIST = [200, 1200, 2500]
STEPS = 100
SAMPLES_PER_CURVE = 10
BRUSHES_PER_PIXEL = 20
LENGTH_SCALE = 1.1
WIDTH_SCALE = 0.1
CANVAS_COLOR = "gray"

CONTENT_WEIGHT = 1.0
STYLE_WEIGHT = 3.0
TV_WEIGHT = 0.008
CURV_WEIGHT = 4.0
LR_GEOM = 1e-1
LR_COLOR = 1e-2

torch.manual_seed(42)
device = pick_device()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Device:", device)
print("Content dir:", CONTENT_DIR)
print("Style manifest:", STYLE_MANIFEST)
print("Output dir:", OUT_DIR)

Root: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer
Device: mps
Content dir: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/images/content
Style manifest: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/images/style_manifest.csv
Output dir: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/results/experiments_90_fixed100_notebook


In [2]:
content_all = sorted([
    p for p in CONTENT_DIR.iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
])
style_all = pd.read_csv(STYLE_MANIFEST)

content_files = content_all[:10]
style_df = style_all.head(3)

print("#content images (selected):", len(content_files))
print("#styles (selected):", len(style_df))
display(style_df)

assert len(content_files) == 10, f"Expected 10 content images, got {len(content_files)}"
assert len(style_df) == 3, f"Expected 3 styles, got {len(style_df)}"

#content images (selected): 10
#styles (selected): 3


,style_path,artist_slug
0,images/style/starry_night.jpg,vincent-van-gogh
1,images/style/water-lilies-monet.jpg,claude-monet
2,images/style/still-life-cezanne.jpg,paul-cezanne


In [3]:
def run_one_pair(content_path: Path, style_path: Path, target_artist: str, num_strokes: int, steps: int):
    content_img = image_loader(str(content_path), IMG_SIZE, device)
    style_img = image_loader(str(style_path), 224, device)
    _, _, H, W = content_img.shape

    vgg_loss = StyleTransferLosses(
        str(VGG_WEIGHTS),
        content_img,
        style_img,
        ["conv4_2", "conv5_2"],
        ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"],
        scale_by_y=True,
    ).to(device).eval()

    content_np = content_img[0].permute(1, 2, 0).cpu().numpy()
    renderer = BrushStrokeRenderer(
        H, W,
        num_strokes=num_strokes,
        samples_per_curve=SAMPLES_PER_CURVE,
        strokes_per_pixel=BRUSHES_PER_PIXEL,
        canvas_color=CANVAS_COLOR,
        length_scale=LENGTH_SCALE,
        width_scale=WIDTH_SCALE,
        content_img=content_np,
    ).to(device)

    optim_geom = optim.Adam(
        [renderer.location, renderer.curve_s, renderer.curve_e, renderer.curve_c, renderer.width],
        lr=LR_GEOM,
    )
    optim_color = optim.Adam([renderer.color], lr=LR_COLOR)

    curves = {"content": [], "style": [], "tv": [], "curvature": [], "total": []}

    for _step in range(1, steps + 1):
        optim_geom.zero_grad()
        optim_color.zero_grad()

        canvas = renderer()
        canvas_img = canvas.unsqueeze(0).permute(0, 3, 1, 2).contiguous()

        content_loss, style_loss = vgg_loss(canvas_img)
        content_loss = content_loss * CONTENT_WEIGHT
        style_loss = style_loss * STYLE_WEIGHT
        tv_loss = TV_WEIGHT * total_variation_loss(renderer.location, renderer.curve_s, renderer.curve_e, K=10)
        curv_loss = CURV_WEIGHT * curvature_loss(renderer.curve_s, renderer.curve_e, renderer.curve_c)
        total_loss = content_loss + style_loss + tv_loss + curv_loss

        total_loss.backward(
            inputs=[renderer.location, renderer.curve_s, renderer.curve_e, renderer.curve_c, renderer.width],
            retain_graph=True,
        )
        optim_geom.step()

        style_loss.backward(inputs=[renderer.color])
        optim_color.step()

        curves["content"].append(float(content_loss.item()))
        curves["style"].append(float(style_loss.item()))
        curves["tv"].append(float(tv_loss.item()))
        curves["curvature"].append(float(curv_loss.item()))
        curves["total"].append(float(total_loss.item()))

    with torch.no_grad():
        final_canvas = renderer()
        final_img = final_canvas.unsqueeze(0).permute(0, 3, 1, 2).contiguous()

    mse = float(torch.mean((final_img - content_img) ** 2).item())
    return final_img.detach(), mse, curves

In [4]:
metrics_path = OUT_DIR / "metrics.csv"
fieldnames = [
    "content_image", "style_image", "target_artist", "num_strokes", "steps", "runtime_sec",
    "content_loss", "style_loss", "tv_loss", "curvature_loss", "total_loss", "reconstruction_mse",
    "deception_rate", "deception_correct", "deception_total", "output_dir",
]

# Stage 1: Renderer code for image generation
with metrics_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for content_path in content_files:
        for _, style_row in style_df.iterrows():
            style_path = Path(style_row["style_path"])
            if not style_path.is_absolute():
                style_path = (ROOT / style_path).resolve()
            target_artist = style_row["artist_slug"]
            if not style_path.exists():
                print("Skipping missing style image:", style_path)
                continue

            for n in NUM_STROKES_LIST:
                run_name = f"{content_path.stem}__{style_path.stem}__n{n}__s{STEPS}"
                run_dir = OUT_DIR / run_name
                run_dir.mkdir(parents=True, exist_ok=True)

                t0 = time.time()
                final_img, mse, curves = run_one_pair(content_path, style_path, target_artist, n, STEPS)
                runtime_sec = time.time() - t0

                save_image(final_img, run_dir / "brushstroke_result.png")

                # Save training curves for loss benchmarking.
                plt.figure(figsize=(10, 6))
                for key, vals in curves.items():
                    plt.plot(vals, label=key)
                plt.xlabel("Step")
                plt.ylabel("Loss")
                plt.title(f"Training Curves: {run_name}")
                plt.legend()
                plt.grid(alpha=0.3)
                plt.tight_layout()
                plt.savefig(run_dir / "training_loss_curves.png", dpi=150)
                plt.close()

                writer.writerow({
                    "content_image": str(content_path),
                    "style_image": str(style_path),
                    "target_artist": target_artist,
                    "num_strokes": n,
                    "steps": STEPS,
                    "runtime_sec": runtime_sec,
                    "content_loss": curves["content"][-1],
                    "style_loss": curves["style"][-1],
                    "tv_loss": curves["tv"][-1],
                    "curvature_loss": curves["curvature"][-1],
                    "total_loss": curves["total"][-1],
                    "reconstruction_mse": mse,
                    "deception_rate": "",
                    "deception_correct": "",
                    "deception_total": "",
                    "output_dir": str(run_dir),
                })
                f.flush()
                print("Generated:", run_name)

print("Generation finished. Saved metrics:", metrics_path)

VGG19 weights loaded.
Generated: car__starry_night__n200__s100
VGG19 weights loaded.
Generated: car__starry_night__n1200__s100
VGG19 weights loaded.
Generated: car__starry_night__n2500__s100
VGG19 weights loaded.
Generated: car__water-lilies-monet__n200__s100
VGG19 weights loaded.
Generated: car__water-lilies-monet__n1200__s100
VGG19 weights loaded.
Generated: car__water-lilies-monet__n2500__s100
VGG19 weights loaded.
Generated: car__still-life-cezanne__n200__s100
VGG19 weights loaded.
Generated: car__still-life-cezanne__n1200__s100
VGG19 weights loaded.
Generated: car__still-life-cezanne__n2500__s100
VGG19 weights loaded.
Generated: city_night__starry_night__n200__s100
VGG19 weights loaded.
Generated: city_night__starry_night__n1200__s100
VGG19 weights loaded.
Generated: city_night__starry_night__n2500__s100
VGG19 weights loaded.
Generated: city_night__water-lilies-monet__n200__s100
VGG19 weights loaded.
Generated: city_night__water-lilies-monet__n1200__s100
VGG19 weights loaded.
Gene

In [5]:
# Stage 2: Deception evaluation
metrics = pd.read_csv(metrics_path)

try:
    _, split_hdf5_path, _ = get_deception_paths()
    deception_model = load_deception_model()
    artist_labels = get_artist_labels(split_hdf5_path)
    print(f"Deception model ready with {len(artist_labels)} artist slugs.")

    for i, row in metrics.iterrows():
        output_dir = Path(row["output_dir"])
        pred_img = output_dir / "brushstroke_result.png"
        if not pred_img.exists():
            continue

        try:
            rate, details = compute_deception_rate(
                deception_model,
                [str(pred_img)],
                target_artist=row["target_artist"],
                split_hdf5_path=split_hdf5_path,
                artist_labels=artist_labels,
            )
            metrics.at[i, "deception_rate"] = float(rate)
            metrics.at[i, "deception_correct"] = int(details["correct"])
            metrics.at[i, "deception_total"] = int(details["total"])
        except ValueError as err:
            print("Deception skip:", output_dir.name, "->", err)
except FileNotFoundError:
    print("Deception assets not found. Keeping deception columns empty.")

metrics.to_csv(metrics_path, index=False)

for c in ["num_strokes", "steps", "runtime_sec", "reconstruction_mse", "total_loss", "deception_rate"]:
    metrics[c] = pd.to_numeric(metrics[c], errors="coerce")

summary = (
    metrics.groupby(["num_strokes", "steps"], as_index=False)
    .agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        mean_reconstruction_mse=("reconstruction_mse", "mean"),
        mean_total_loss=("total_loss", "mean"),
        mean_deception_rate=("deception_rate", "mean"),
        num_runs=("output_dir", "count"),
    )
    .sort_values(["num_strokes", "steps"])
)
summary.to_csv(OUT_DIR / "aggregate_summary.csv", index=False)

# Save benchmark plots.
curve = metrics.groupby("num_strokes", as_index=False)["reconstruction_mse"].mean().sort_values("num_strokes")
plt.figure(figsize=(6, 4))
plt.plot(curve["num_strokes"], curve["reconstruction_mse"], marker="o")
plt.xlabel("Number of strokes")
plt.ylabel("Mean reconstruction MSE")
plt.title("Reconstruction MSE vs Stroke Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "mse_vs_strokes.png", dpi=150)
plt.close()

curve = metrics.groupby("num_strokes", as_index=False)["deception_rate"].mean().sort_values("num_strokes")
plt.figure(figsize=(6, 4))
plt.plot(curve["num_strokes"], curve["deception_rate"], marker="o")
plt.xlabel("Number of strokes")
plt.ylabel("Mean deception rate")
plt.title("Deception Rate vs Stroke Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "deception_vs_strokes.png", dpi=150)
plt.close()

curve = metrics.groupby("steps", as_index=False)["runtime_sec"].mean().sort_values("steps")
plt.figure(figsize=(6, 4))
plt.plot(curve["steps"], curve["runtime_sec"], marker="o")
plt.xlabel("Optimization steps")
plt.ylabel("Mean runtime (sec)")
plt.title("Runtime vs Optimization Steps")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "runtime_vs_steps.png", dpi=150)
plt.close()

print("Metrics rows:", len(metrics))
expected = 10 * 3 * len(NUM_STROKES_LIST)
print("Expected rows:", expected)
if len(metrics) != expected:
    print("Warning: row count differs from expected. Check missing images/style paths.")

display(summary)
display(metrics.head(5))

Deception score model loaded from TF checkpoint: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/deception_score_vgg/model.ckpt-790000
Deception model ready with 689 artist slugs.
Metrics rows: 90
Expected rows: 90


,num_strokes,steps,mean_runtime_sec,mean_reconstruction_mse,mean_total_loss,mean_deception_rate,num_runs
0,200,100,65.266749,0.040124,263684.175911,0.000000,30
1,1200,100,149.435724,0.039157,1647.378271,0.233333,30
2,2500,100,280.360147,0.038109,230.618829,0.300000,30


,content_image,style_image,target_artist,num_strokes,steps,runtime_sec,content_loss,style_loss,tv_loss,curvature_loss,total_loss,reconstruction_mse,deception_rate,deception_correct,deception_total,output_dir
0,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,200,100,75.188750,11.887683,14.632065,772388.250000,0.188571,772414.937500,0.057040,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
1,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,1200,100,108.065675,11.219572,4.209731,6028.085449,0.418294,6043.933105,0.054662,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
2,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,2500,100,132.091224,10.734131,3.409535,867.734436,0.423820,882.301941,0.054580,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
3,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,claude-monet,200,100,80.276258,11.942498,3.410959,774435.062500,0.109004,774450.562500,0.049316,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
4,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,claude-monet,1200,100,107.922762,11.334877,2.148756,6022.341309,0.201375,6036.025879,0.054735,1.0,1.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
